In [ ]:
import numpy as np
import pandas as pd
from functools import reduce

In [ ]:
#Read in standardized proteins
rs_olink = pd.read_csv('Data/RS/RS_olink_imputed.csv')

#calculate age as time between birth and blood-sampling and scale
rs_age = pd.read_spss('Data/RS/RoterdamStudy_Basics2014.sav')
rs_bld = pd.read_spss('Data/RS/e5_(5)_BLDAFNAM_(10-jul-2017).sav')
rs_bld = rs_bld.rename(columns={'e5_2686': 'blooddate'})
rs_bld = rs_bld[['ergoid', 'blooddate']]

rs_age = rs_age.merge(rs_bld, how = 'inner')
rs_age['age_raw'] = (pd.to_datetime(rs_age['blooddate']) - pd.to_datetime(rs_age['date_of_birth']) )/ pd.Timedelta('365 days')

#Make sex same as UKBB
rs_age['sex'] = np.where(rs_age['sex'] == 'male', 1, 0)

#Merge with bigger file
rs = pd.merge(rs_olink, rs_age[['ergoid','age_raw', 'blooddate', 'sex']], how = 'left', left_on='SampleID', right_on='ergoid')
rs.drop('SampleID', axis=1, inplace = True)

#save for later usage
rsnn = rs

In [ ]:
# Function to make one big dataframe
def load_and_prepare(file_path, index_col='ergoid', join_columns=None, rename_columns=None, calc_columns=None):
    # Load data
    if file_path.endswith('.sav'):
        df = pd.read_spss(file_path)
    elif file_path.endswith('.csv') or file_path.endswith('.txt'):
        df = pd.read_csv(file_path, sep="\t" if file_path.endswith('.txt') else ",")
    else:
        raise ValueError("Unsupported file format")
    
    # Set index
    df.set_index(index_col, inplace=True)
    
    # Rename columns
    if rename_columns:
        df.rename(columns=rename_columns, inplace=True)
    
    # Calculate new columns
    if calc_columns:
        for col_name, calc_func in calc_columns.items():
            df[col_name] = calc_func(df, age)  # Pass `age` dataframe for calculations if needed
    
    # Select specified columns
    if join_columns:
        df = df[join_columns]
    
    return df

# Define datasets with their processing instructions
datasets = [
    {
        "file_path": "Data/RS/fp_VitalStatus_2024-26.sav",
        "calc_columns": {
            #Make follow-up time as time between blood sampling and censoring
            #Censor age as time between blood sampling and censoring + chronological age to account for left censoring
            "followup": lambda df, age: (pd.to_datetime(df['fp_censordate']) - pd.to_datetime(age['blooddate'])) / pd.Timedelta('365 days'),
            "censorage": lambda df, age: (pd.to_datetime(df['fp_censordate']) - pd.to_datetime(age['blooddate'])) / pd.Timedelta('365 days') + age['age_raw'],
            "died": lambda df, _: np.where(df['fp_vitalstatus'] == 'dead', 1, 0)
        },
        "join_columns": ["followup", "censorage", "died"]
    },
    {
        "file_path": "Data/RS/RS_CHD.csv",
        "calc_columns": {
            #Make follow-up time as time between blood sampling and censoring
            #Censor age as time between blood sampling and censoring + chronological age to account for left censoring
            "CVD_followup": lambda df, age: (pd.to_datetime(df['enddate_CHD']) - pd.to_datetime(age['blooddate'])) / pd.Timedelta('365 days'),
            "CVD_age": lambda df, age: (pd.to_datetime(df['enddate_CHD']) - pd.to_datetime(age['blooddate'])) / pd.Timedelta('365 days') + age['age_raw']
        },
        "rename_columns": {"inc_CHD": "CVD_inc", "prev_CHD":"CVD_prev"},
        "join_columns": ["CVD_followup", "CVD_age", "CVD_inc", "CVD_prev"]
    },
    
    {"file_path": "Data/RS/Frailty_E5.txt", 
     "join_columns": ["FI", "FRIED", "TFI", "CHARGE", "MPI", "TFIC", "FRIED5"]},
    {
        "file_path": "Data/RS/e5_(5)_ANTHROPO_(10-dec-2015).sav",
        #Calculate bmi as weight (kg)/height (m)^2
        "calc_columns": {"bmi": lambda df, _: df['e5_230'].astype(float) / (df['e5_229'].astype(float) / 100) ** 2},
        "join_columns": ["bmi"]
    },
    {
        "file_path": "Data/RS/e5_intvw_SMOKING_(22-mar-2021).sav",
        "calc_columns": {
            #Make smoking cigarettes binary but only for people that answered.
         "smokingyn": lambda df, _: np.where(
            (df['e5_EILF6'] == 'ja') & (~df['e5_EILF6'].isin(['weet het niet', 'geen antwoord'])), 1, 0)},
        "join_columns": ["smokingyn"]
    },
    {
        "file_path": "Data/RS/HT2018_analysisfile_(15-may-2018).sav",
        "calc_columns": {"hbp": lambda df, _: np.where(df['e5_HT2018'] == 'yes', 1, 0)},
        "join_columns": ["hbp"]
    },
    {
        "file_path": "Data/RS/Education RS-I-II-III (UNESCO class)_(12-MAR-2015).sav",
         #Use more interpretable name for ses
        #"rename_columns": {"ses_UNESCO_recoded": "ses"},
        "join_columns": ["ses_UNESCO_recoded"]
    },
    {"file_path": "Data/RS/RSI-5_RSII-3_RSIII-2_handgrip.sav", 
     "join_columns": ["max_handgrip"]},
    {"file_path": "Data/RS/e5_(5)_LAB_(29-aug-2014).sav",
     #Use more interpretable names for cell counts
     "rename_columns": {"e5_12794": "lymph_perc", "e5_12801" : "mono_perc"},
     "join_columns": ["lymph_perc", "mono_perc"]
    },
    {"file_path" : "Data/RS/e5_intvw_PAINPERC_(25-apr-2019).sav",
     "calc_columns": {
          #Make variable poor health for people who believed their health was poor compared to peers but not when didn't know or didn't answer
         "poor_health": lambda df, _: np.where(
            (df['e5_EIGEZLFT'] == 'slechter') & (~df['e5_EIGEZLFT'].isin(['Weet het niet', 'missing'])), 1, 0)},
     "join_columns": ["poor_health"]}
        
]

#Age 
rs.set_index('ergoid', inplace = True)
age = rs[['age_raw', 'blooddate', 'sex']]

# Want to left_join with age so start with that
cov_data = age.copy()

# Iteratively merge each dataset
for dataset in datasets:
    df = load_and_prepare(
        file_path=dataset["file_path"],
        join_columns=dataset.get("join_columns"),
        rename_columns=dataset.get("rename_columns"),
        calc_columns=dataset.get("calc_columns")
    )
    cov_data = cov_data.merge(df, how='left', left_index=True, right_index=True)

cov_data['lymph_perc'] = cov_data['lymph_perc'].astype('float')
cov_data['mono_perc'] = cov_data['mono_perc'].astype('float')

# SES disturbs the rest try differently
ses_mapping = {
    'higher vocational education OR university': 0,
    'lower/intermediate general education OR lower vocational education': 1,
    'intermediate vocational education OR higher general education': 2,
    'primary education': 3
}

# Apply the mapping to the ses column
cov_data['ses'] = cov_data['ses_UNESCO_recoded'].map(ses_mapping)


In [ ]:
cov_data.index.isin(rs.index).sum()

In [ ]:
#Make one giant dataframe
dfs_to_merge = [cov_data]
own_df = reduce(lambda left, right: pd.merge(left, right, left_index=True, right_index=True), dfs_to_merge)
list(own_df.columns)

In [ ]:
cov_data['sex'].describe()

In [ ]:
def summarize_data(df, numeric_cols, categorical_conditions):
    summary = {}

    # Descriptive statistics for numeric columns
    if numeric_cols:
        summary['numeric_stats'] = df[numeric_cols].agg(['mean', 'std', 'median', lambda x: x.quantile(0.25), lambda x: x.quantile(0.75)]).T
        summary['numeric_stats'].columns = ['mean', 'std', 'median', 'Q1', 'Q3']

    # N and Proportions for categorical conditions
    categorical_summary = {}
    for col, condition in categorical_conditions.items():
        valid_values = df[col].dropna()  # Exclude NaNs from the denominator
        count = (valid_values == condition).sum()
        total_valid = len(valid_values)

        if total_valid > 0:
            proportion = (count / total_valid) * 100
        else:
            proportion = np.nan  # If no valid values, return NaN

        # Get NaN count and percentage
        na_count = df[col].isna().sum()
        na_percentage = (na_count / df.shape[0]) * 100 if df.shape[0] > 0 else np.nan

        # Value counts and percentages for each category
        value_counts = df[col].value_counts(dropna=False)  # Get all categories, including NaN
        value_counts_percentage = (value_counts / df.shape[0]) * 100  # Calculate percentages

        categorical_summary[col] = {
            'count': count,
            'proportion': proportion,
            'na_count': na_count,
            'na_percentage': na_percentage,
            'value_counts': value_counts,
            'value_counts_percentage': value_counts_percentage
        }

    summary['categorical_summary'] = categorical_summary

    return summary


# Define numeric columns and categorical conditions
numeric_cols = ['age_raw', 'bmi', 'lymph_perc', 'mono_perc', 'FI', 'max_handgrip', 'followup', 'CVD_followup']
categorical_conditions_met = {
    'sex': 0,
    'smokingyn': 1,
    'died' : 1,
    'poor_health': 1,
    'hbp': 1,
    'CVD_inc' : 1,
    'CVD_prev' : 1,
    'ses': 0,  
    'ses': 1,  
    'ses': 2,  
    'ses': 3  
    
}
categorical_conditions_not_met = categorical_conditions_met.copy()

# Summarize df_met
summary_met = summarize_data(cov_data, numeric_cols, categorical_conditions_met)

# Summarize df_not_met
summary_not_met = summarize_data(cov_data, numeric_cols, categorical_conditions_not_met)

# Print summaries
def print_summary(summary, name):
    print(f"\nSummary for {name}:")
    print("Numeric Stats (Mean and Std):\n", summary['numeric_stats'])
    print("\nCategorical Summary:")
    
    for col, stats in summary['categorical_summary'].items():
        print(f"{col}: Count = {stats['count']} ({round(stats['proportion'], 1)}), "
              f"NA = {stats['na_count']} ({round(stats['na_percentage'], 1)})")
        
        value_counts_str = ", ".join(
            f"{value}: {stats['value_counts'][value]} ({round(stats['value_counts_percentage'][value], 1)})"
            for value in stats['value_counts'].index
        )
        print(f"  {value_counts_str}")

print_summary(summary_met, "df_met")